# YOLO26 Object Detection on AWS Trainium2 with Neuron SDK

This notebook demonstrates compiling and running [Ultralytics YOLO26](https://github.com/ultralytics/ultralytics) detection models on AWS Trainium2 (trn2) using `torch_neuronx.trace()`. All 5 detection variants (n/s/m/l/x) are covered, along with data-parallel scaling across NeuronCores.

| | |
|---|---|
| **Instance** | trn2.3xlarge (1 Trainium2 chip, 4/8 NeuronCores) |
| **SDK** | Neuron SDK 2.29 (DLAMI 20260410) |
| **Time to complete** | ~15 minutes (compilation + benchmarks) |
| **Prerequisites** | trn2 instance with Neuron DLAMI |

**Quick Start:**
```bash
source /opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/bin/activate
pip install ultralytics
jupyter lab --no-browser --port=8888
```

### Key Results

| Variant | Params | Neuron Peak (LNC=1, DP=8) | A10G Compiled FP16 | Neuron Speedup |
|---------|--------|--------------------------|--------------------|-----------|
| YOLO26n | 2.4M | 272 img/s | 2,166 img/s | 0.13x |
| YOLO26s | 10.0M | **1,523 img/s** | 1,065 img/s | **1.43x** |
| YOLO26m | 21.9M | **1,267 img/s** | 474 img/s | **2.67x** |
| YOLO26l | 26.3M | **1,093 img/s** | 371 img/s | **2.95x** |
| YOLO26x | 58.9M | **876 img/s** | 195 img/s | **4.49x** |

---
## Step 0: Setup & Environment Check

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"])

In [ ]:
import os
import time
import json

import torch
import torch_neuronx
import numpy as np
from ultralytics import YOLO
from ultralytics.nn.modules.block import C2f

print(f"PyTorch:       {torch.__version__}")
print(f"torch-neuronx: {torch_neuronx.__version__}")
print(f"Neuron SDK:    {torch_neuronx.__version__}")

In [ ]:
# Show available NeuronCores
subprocess.run(["neuron-ls"])

# Check LNC mode
lnc = os.environ.get("NEURON_LOGICAL_NC_CONFIG", "2")
print(f"\nLNC mode: {lnc}")
num_cores = 8 if lnc == "1" else 4
print(f"Available logical cores: {num_cores}")

---
## Step 1: Download Model Weights

In [ ]:
VARIANTS = ["n", "s", "m", "l", "x"]
WEIGHT_DIR = "."

for v in VARIANTS:
    wp = f"yolo26{v}.pt"
    if not os.path.exists(wp):
        print(f"Downloading {wp}...")
        _ = YOLO(wp)  # ultralytics auto-downloads
    else:
        print(f"{wp} already exists")

---
## Step 2: Model Preparation & Tracing Recipe

YOLO26 requires specific preparation before `torch_neuronx.trace()`:

1. **`end2end=False`** before `fuse()` -- disables `topk` (unsupported on Neuron)
2. **`fuse()`** -- merges BatchNorm into Conv2d, removes training branches
3. **`export=True`** -- single-tensor output for clean tracing
4. **`C2f.forward_split`** -- uses `split()` instead of `chunk()` for tracing
5. **Dry run** -- populates anchor/stride caches
6. **BF16 for m/l/x** -- FP32 exceeds SB allocation for larger variants

Output: raw `[B, 84, 8400]` tensor (4 bbox coords + 80 COCO class scores per anchor). Postprocessing (topk, NMS) happens on CPU.

In [ ]:
def prepare_yolo26(weight_path, dtype=torch.float32):
    """Prepare YOLO26 detection model for Neuron tracing."""
    model = YOLO(weight_path)
    pytorch_model = model.model.eval()

    # Disable end2end BEFORE fuse (topk not supported on Neuron)
    detect = pytorch_model.model[-1]
    detect.end2end = False

    pytorch_model = pytorch_model.fuse(verbose=False)

    for m in pytorch_model.modules():
        if hasattr(m, "export"):
            m.export = True
        if hasattr(m, "dynamic"):
            m.dynamic = False
        if hasattr(m, "format"):
            m.format = "torchscript"
        if hasattr(m, "shape"):
            m.shape = None
        if isinstance(m, C2f):
            m.forward = m.forward_split

    if dtype != torch.float32:
        pytorch_model = pytorch_model.to(dtype)

    return pytorch_model


# Variant configurations: n/s use FP32, m/l/x require BF16
VARIANT_CONFIG = {
    "n": {"dtype": torch.float32, "dtype_name": "fp32"},
    "s": {"dtype": torch.float32, "dtype_name": "fp32"},
    "m": {"dtype": torch.bfloat16, "dtype_name": "bf16"},
    "l": {"dtype": torch.bfloat16, "dtype_name": "bf16"},
    "x": {"dtype": torch.bfloat16, "dtype_name": "bf16"},
}

print("Variant configuration:")
for v, cfg in VARIANT_CONFIG.items():
    print(f"  yolo26{v}: {cfg['dtype_name']}")

---
## Step 3: Compile All Variants

Compile all 5 detection variants for Neuron. Each compilation takes 30-70 seconds.

In [ ]:
COMPILE_DIR = "compiled"
os.makedirs(COMPILE_DIR, exist_ok=True)
BS = 1  # Batch size for initial compilation

# Check LNC mode for compiler args
compiler_args = []
if lnc == "1":
    compiler_args = ["--lnc", "1"]
    print("Using --lnc 1 compiler flag for LNC=1 mode")

compile_results = {}

for v in VARIANTS:
    cfg = VARIANT_CONFIG[v]
    neff_path = os.path.join(COMPILE_DIR, f"yolo26{v}_{cfg['dtype_name']}_bs{BS}.pt")

    if os.path.exists(neff_path):
        print(f"yolo26{v}: already compiled at {neff_path}")
        compile_results[v] = {"status": "cached", "path": neff_path}
        continue

    print(f"\nCompiling yolo26{v} ({cfg['dtype_name']}, BS={BS})...")
    model = prepare_yolo26(f"yolo26{v}.pt", dtype=cfg["dtype"])

    dummy = torch.randn(BS, 3, 640, 640, dtype=cfg["dtype"])
    with torch.no_grad():
        _ = model(dummy)  # dry run for anchors

    t0 = time.time()
    traced = torch_neuronx.trace(model, dummy, compiler_args=compiler_args)
    compile_time = time.time() - t0

    torch.jit.save(traced, neff_path)
    neff_size = os.path.getsize(neff_path) / (1024 * 1024)

    print(f"  Compiled in {compile_time:.1f}s, NEFF: {neff_size:.1f} MB")
    compile_results[v] = {
        "status": "compiled",
        "path": neff_path,
        "compile_time_s": round(compile_time, 1),
        "neff_size_mb": round(neff_size, 1),
    }
    del model, traced

print("\nCompilation summary:")
for v, r in compile_results.items():
    ct = r.get('compile_time_s', 'cached')
    ns = r.get('neff_size_mb', '?')
    print(f"  yolo26{v}: {ct}s, {ns} MB")

---
## Step 4: Accuracy Validation

Compare Neuron output against CPU reference using cosine similarity.

In [ ]:
print(f"{'Variant':<10} {'Dtype':<6} {'CosSim':<10} {'MaxErr':<12} {'MeanErr':<12} {'Neuron NaN':<10}")
print("-" * 60)

for v in VARIANTS:
    cfg = VARIANT_CONFIG[v]
    neff_path = compile_results[v]["path"]

    # CPU reference
    model_cpu = prepare_yolo26(f"yolo26{v}.pt", dtype=cfg["dtype"])
    dummy = torch.randn(1, 3, 640, 640, dtype=cfg["dtype"])
    with torch.no_grad():
        cpu_out = model_cpu(dummy)

    # Neuron inference
    traced = torch.jit.load(neff_path)
    with torch.no_grad():
        nrn_out = traced(dummy)

    # Compare
    cpu_flat = cpu_out.flatten().float()
    nrn_flat = nrn_out.flatten().float()
    cossim = torch.nn.functional.cosine_similarity(cpu_flat.unsqueeze(0), nrn_flat.unsqueeze(0)).item()
    max_err = (cpu_flat - nrn_flat).abs().max().item()
    mean_err = (cpu_flat - nrn_flat).abs().mean().item()
    has_nan = torch.isnan(nrn_out).any().item()

    print(f"yolo26{v:<4} {cfg['dtype_name']:<6} {cossim:<10.6f} {max_err:<12.6f} {mean_err:<12.6f} {has_nan}")

    del model_cpu, traced

---
## Step 5: Single-Core Latency Benchmark

In [ ]:
WARMUP = 20
ITERATIONS = 100

print(f"{'Variant':<10} {'Dtype':<6} {'p50 (ms)':<12} {'p95 (ms)':<12} {'p99 (ms)':<12} {'img/s':<10}")
print("-" * 62)

single_core_results = {}

for v in VARIANTS:
    cfg = VARIANT_CONFIG[v]
    traced = torch.jit.load(compile_results[v]["path"])
    dummy = torch.randn(1, 3, 640, 640, dtype=cfg["dtype"])

    # Warmup
    for _ in range(WARMUP):
        with torch.no_grad():
            traced(dummy)

    # Measure
    latencies = []
    for _ in range(ITERATIONS):
        t0 = time.time()
        with torch.no_grad():
            traced(dummy)
        latencies.append((time.time() - t0) * 1000)

    lat = np.array(sorted(latencies))
    p50 = np.percentile(lat, 50)
    p95 = np.percentile(lat, 95)
    p99 = np.percentile(lat, 99)
    throughput = 1000.0 / p50

    print(f"yolo26{v:<4} {cfg['dtype_name']:<6} {p50:<12.2f} {p95:<12.2f} {p99:<12.2f} {throughput:<10.1f}")
    single_core_results[v] = {"p50": p50, "throughput": throughput}

    del traced

---
## Step 6: Data-Parallel Scaling

Scale inference across all NeuronCores using `torch_neuronx.DataParallel`. Each core runs an independent model replica, multiplying throughput.

On trn2.3xlarge:
- **LNC=2** (default): 4 logical cores, DP=4
- **LNC=1**: 8 logical cores, DP=8 (recommended for YOLO26)

In [ ]:
# Pick yolo26s (best overall balance) for the DP demo
DP_WARMUP = 10
DP_ITERATIONS = 50
DEMO_VARIANT = "s"
cfg = VARIANT_CONFIG[DEMO_VARIANT]
BATCH_SIZES = [1, 4, 8]

print(f"Data-Parallel benchmark: yolo26{DEMO_VARIANT} ({cfg['dtype_name']}), DP={num_cores}")
print(f"{'BS/core':<10} {'Total BS':<10} {'p50 (ms)':<12} {'Throughput':<14} {'vs BS=1':<10}")
print("-" * 56)

dp_results = []

for bs in BATCH_SIZES:
    # Compile for this batch size if not already done
    neff_path = os.path.join(COMPILE_DIR, f"yolo26{DEMO_VARIANT}_{cfg['dtype_name']}_bs{bs}.pt")
    if not os.path.exists(neff_path):
        model = prepare_yolo26(f"yolo26{DEMO_VARIANT}.pt", dtype=cfg["dtype"])
        dummy = torch.randn(bs, 3, 640, 640, dtype=cfg["dtype"])
        with torch.no_grad():
            _ = model(dummy)
        traced = torch_neuronx.trace(model, dummy, compiler_args=compiler_args)
        torch.jit.save(traced, neff_path)
        del model, traced

    # Load and create DataParallel
    traced = torch.jit.load(neff_path)
    model_dp = torch_neuronx.DataParallel(
        traced,
        device_ids=list(range(num_cores)),
        dim=0,
    )

    total_bs = bs * num_cores
    dummy_dp = torch.randn(total_bs, 3, 640, 640, dtype=cfg["dtype"])

    # Warmup
    for _ in range(DP_WARMUP):
        with torch.no_grad():
            model_dp(dummy_dp)

    # Measure
    latencies = []
    for _ in range(DP_ITERATIONS):
        t0 = time.time()
        with torch.no_grad():
            model_dp(dummy_dp)
        latencies.append((time.time() - t0) * 1000)

    lat = np.array(sorted(latencies))
    p50 = np.percentile(lat, 50)
    throughput = total_bs / (p50 / 1000)

    dp_results.append({"bs": bs, "total_bs": total_bs, "p50": p50, "throughput": throughput})

    ratio = throughput / dp_results[0]["throughput"] if dp_results else 1.0
    print(f"{bs:<10} {total_bs:<10} {p50:<12.2f} {throughput:<14.1f} {ratio:<10.2f}x")

    del model_dp, traced

peak = max(dp_results, key=lambda x: x["throughput"])
print(f"\nPeak: {peak['throughput']:.1f} img/s at BS={peak['bs']}/core (total BS={peak['total_bs']})")

---
## Step 7: Multi-Variant DP Benchmark

Benchmark all 5 variants at their optimal batch sizes with full data parallelism.

In [ ]:
# Optimal BS per variant (from Task 008 sweep)
# LNC=1 (DP=8): n=1, s=32, m=32, l=32, x=16
# LNC=2 (DP=4): use smaller BS to reduce compile+benchmark time
if lnc == "1":
    OPTIMAL_BS = {"n": 1, "s": 32, "m": 32, "l": 32, "x": 16}
else:
    OPTIMAL_BS = {"n": 1, "s": 8, "m": 8, "l": 8, "x": 4}

print(f"All variants, DP={num_cores}, optimal batch sizes")
print(f"{'Variant':<10} {'Dtype':<6} {'BS/core':<10} {'Total BS':<10} {'p50 (ms)':<12} {'img/s':<12}")
print("-" * 60)

all_variant_results = {}

for v in VARIANTS:
    cfg = VARIANT_CONFIG[v]
    bs = OPTIMAL_BS[v]
    total_bs = bs * num_cores

    # Compile if needed
    neff_path = os.path.join(COMPILE_DIR, f"yolo26{v}_{cfg['dtype_name']}_bs{bs}.pt")
    if not os.path.exists(neff_path):
        model = prepare_yolo26(f"yolo26{v}.pt", dtype=cfg["dtype"])
        dummy = torch.randn(bs, 3, 640, 640, dtype=cfg["dtype"])
        with torch.no_grad():
            _ = model(dummy)
        traced = torch_neuronx.trace(model, dummy, compiler_args=compiler_args)
        torch.jit.save(traced, neff_path)
        del model, traced

    traced = torch.jit.load(neff_path)
    model_dp = torch_neuronx.DataParallel(
        traced, device_ids=list(range(num_cores)), dim=0
    )

    dummy_dp = torch.randn(total_bs, 3, 640, 640, dtype=cfg["dtype"])

    for _ in range(DP_WARMUP):
        with torch.no_grad():
            model_dp(dummy_dp)

    latencies = []
    for _ in range(DP_ITERATIONS):
        t0 = time.time()
        with torch.no_grad():
            model_dp(dummy_dp)
        latencies.append((time.time() - t0) * 1000)

    lat = np.array(sorted(latencies))
    p50 = np.percentile(lat, 50)
    throughput = total_bs / (p50 / 1000)

    print(f"yolo26{v:<4} {cfg['dtype_name']:<6} {bs:<10} {total_bs:<10} {p50:<12.2f} {throughput:<12.1f}")
    all_variant_results[v] = {"throughput": throughput, "p50": p50, "bs": bs}

    del model_dp, traced

---
## Step 8: GPU Comparison

Comparison against NVIDIA A10G (g5.xlarge) with `torch.compile(mode='reduce-overhead')` + FP16.

| Variant | Neuron trn2 (DP=8) | A10G Compiled FP16 | Neuron / A10G | Cost: Neuron $/M | Cost: A10G $/M |
|---------|-------------------|--------------------|---------------|-----------------|----------------|
| YOLO26n | 272 img/s | **2,166 img/s** | 0.13x | $3.87 | $0.13 |
| YOLO26s | **1,523 img/s** | 1,065 img/s | **1.43x** | $0.69 | $0.26 |
| YOLO26m | **1,267 img/s** | 474 img/s | **2.67x** | $0.83 | $0.59 |
| YOLO26l | **1,093 img/s** | 371 img/s | **2.95x** | $0.96 | $0.75 |
| YOLO26x | **876 img/s** | 195 img/s | **4.49x** | $1.20 | $1.43 |

*Pricing: trn2.3xlarge ~$3.79/hr, g5.xlarge $1.006/hr (us-east-1 on-demand)*

**Neuron wins 1.4-4.5x on throughput for s/m/l/x variants.** Cost-advantaged on YOLO26x ($1.20 vs $1.43 per million images). The advantage grows with model size.

---
## Step 9: Additional Task Variants

The same tracing recipe works for segmentation, pose, and OBB variants:

In [ ]:
ADDITIONAL_VARIANTS = {
    "pose": {"weight": "yolo26s-pose.pt", "input": (1, 3, 640, 640)},
    "obb": {"weight": "yolo26s-obb.pt", "input": (1, 3, 640, 640)},
}

for task_name, task_cfg in ADDITIONAL_VARIANTS.items():
    print(f"\n--- {task_name.upper()} ---")
    wp = task_cfg["weight"]
    if not os.path.exists(wp):
        _ = YOLO(wp)

    model = prepare_yolo26(wp)
    dummy = torch.randn(*task_cfg["input"])
    with torch.no_grad():
        cpu_out = model(dummy)

    neff_path = os.path.join(COMPILE_DIR, f"{wp.replace('.pt', '_neuron.pt')}")
    if not os.path.exists(neff_path):
        traced = torch_neuronx.trace(model, dummy, compiler_args=compiler_args)
        torch.jit.save(traced, neff_path)
    else:
        traced = torch.jit.load(neff_path)

    with torch.no_grad():
        nrn_out = traced(dummy)

    cossim = torch.nn.functional.cosine_similarity(
        cpu_out.flatten().float().unsqueeze(0),
        nrn_out.flatten().float().unsqueeze(0)
    ).item()

    # Latency
    for _ in range(10):
        with torch.no_grad():
            traced(dummy)
    latencies = []
    for _ in range(50):
        t0 = time.time()
        with torch.no_grad():
            traced(dummy)
        latencies.append((time.time() - t0) * 1000)
    p50 = np.median(latencies)

    print(f"  Output: {list(nrn_out.shape) if isinstance(nrn_out, torch.Tensor) else [list(t.shape) for t in nrn_out]}")
    print(f"  CosSim: {cossim:.6f}")
    print(f"  Latency: {p50:.2f} ms ({1000/p50:.1f} img/s)")

    del model, traced

---
## Summary

This notebook demonstrated:

- **All 5 YOLO26 detection variants** (n/s/m/l/x) compile and run on Neuron
- **Same tracing recipe** works across detection, pose, and OBB task heads
- **Data-parallel scaling** across 4-8 NeuronCores for throughput multiplication
- **Neuron outperforms A10G** by 1.4-4.5x on s/m/l/x variants (compiled GPU baseline)
- **Key tracing requirements**: `end2end=False` (topk unsupported), BF16 for m/l/x (FP32 SB overflow), no `--auto-cast` flags (NaN for Conv2d models)

### Recommended Configurations

| Variant | LNC | DP | BS/core | Peak img/s |
|---------|-----|-----|---------|------------|
| YOLO26n | 1 | 8 | 1 (FP32) | 272 |
| YOLO26s | 1 | 8 | 32 (BF16) | 1,523 |
| YOLO26m | 1 | 8 | 32 (BF16) | 1,267 |
| YOLO26l | 1 | 8 | 32 (BF16) | 1,093 |
| YOLO26x | 1 | 8 | 16 (BF16) | 876 |